# Chapter 16 – Worked examples

This notebook runs the examples of [Chapter 16](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/devdev/part-3-power-systems/ch16-dc-power-flow/) live. The code lives in the modules next to it
(`dc_power_flow.py`, `dc_vs_ac.py`, `dc_sparse_solve.py`, `psdf.py`); the cells only call them, so a change in a
module is a change here too. The Octave example (`matpower_dcpf.m`) runs in Octave, and `toop57.py` needs ToOp's
environment; neither runs here.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q colorama==0.4.6 deepdiff==8.6.2 geojson==3.3.0 networkx numpy packaging pandas pandera==0.34.1 scipy tqdm typing_extensions tzdata
%pip install -q --no-deps pandapower==3.5.6

In [ ]:
import dc_power_flow
import dc_sparse_solve
import dc_vs_ac
import numpy as np
import psdf
import scipy.sparse.linalg as spla

## From AC to DC: a three-bus ring

100 MW enter at bus 0 and leave at bus 2. The direct branch 0–2 has $x = 0.2$, the path 0–1–2 has
$x = 0.1 + 0.1 = 0.2$: the two paths share the power equally, 50 MW each, as the inverse of the path reactances
predicts.

In [ ]:
frm, to = np.array([0, 1, 0]), np.array([1, 2, 2])
print(dc_power_flow.dc_power_flow(frm, to, np.array([0.1, 0.1, 0.2]), np.array([100.0, 0.0, -100.0]), slack=0))

## Matrix form on IEEE 14

$B_f = \operatorname{diag}(b) A$ and $B_\text{bus} = A^\top \operatorname{diag}(b) A$ with $b = 1/(x\tau)$, built
as sparse matrices from pandapower's branch table. Every row of $B_\text{bus}$ sums to zero (it is a Laplacian), so
the slack row and column are removed before the solve. The flows equal `pandapower.rundcpp`'s.

In [ ]:
_, table, p, slack = dc_vs_ac.dc_case('case14')
n_bus = int(max(table['from'].max(), table['to'].max())) + 1
b = 1 / (table['x'].to_numpy() * table['tap'].to_numpy())
bbus, bf = dc_sparse_solve.susceptance_matrices(table['from'].to_numpy(), table['to'].to_numpy(), b, n_bus)
print('Bbus', bbus.shape, 'nonzeros', bbus.nnz, '| row sums', np.abs(bbus.sum(axis=1)).max())
keep = np.arange(n_bus) != slack
flows = dc_sparse_solve.flows_many(spla.splu(bbus[keep][:, keep]), bf[:, keep], p[None, :], keep)[0]
print('max |own - rundcpp| =', f'{np.abs(flows - table["p_pandapower"].to_numpy()).max():.1e}', 'MW')

## Phase shifters as equivalent injections

In ToOp's convention the flow on branch $k$ is $b_k(\theta_f - \theta_t + \varphi_k)$: the shift acts as
injections $-b_k\varphi_k$ at $f_k$ and $+b_k\varphi_k$ at $t_k$ plus a direct flow $b_k\varphi_k$. On a four-bus
ring a 5° shift on branch 0 moves 0.145 pu around the loop, $\varphi / \sum x$ for a single loop.

In [ ]:
frm, to = np.array([0, 0, 1, 2]), np.array([1, 2, 3, 3])
x, p4 = np.array([0.1, 0.2, 0.1, 0.2]), np.array([0.8, 0.2, -0.5, -0.5])
phi = np.zeros(4)
phi[0] = np.deg2rad(5.0)
print('without shift:', np.round(dc_power_flow.dc_power_flow(frm, to, x, p4, 0), 4))
print('with 5 deg    :', np.round(psdf.flows_with_shift(frm, to, x, p4, 0, phi), 4))

## The multi-timestep axis

One topology, many injection snapshots: `nodal_injections` has shape `(n_timesteps, n_bus)` in ToOp, and the
flows follow with one factorization. Here 1,000 balanced random snapshots of IEEE 14 in one call.

In [ ]:
rng = np.random.default_rng(16)
snapshots = p + rng.normal(scale=10.0, size=(1000, n_bus))
snapshots[:, slack] -= snapshots.sum(axis=1)
many = dc_sparse_solve.flows_many(spla.splu(bbus[keep][:, keep]), bf[:, keep], snapshots, keep)
print(f'flows shape (n_timesteps, n_branches): {many.shape} | largest flow {np.abs(many).max():.1f} MW')